# Практическая работа 
Прогнозирование финансовых временных рядов на данных Московской биржи

Цель работы  - демонстрация ключевых концепций лекции по прогнозированию временных рядов (разложение на тренд и сезонность, многомасштабные зависимости, метрики оценки) к реальным данным Московской биржи.  

Вам предстоит построить полный пайплайн: от получения данных через `apimoex` до сравнения архитектур N-BEATS и ConvTimeNet.  

## 1. Пример прогнозирования финансового ряда


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Настройка воспроизводимости результатов
torch.manual_seed(42)
np.random.seed(42)


In [ ]:
# ==========================================
# 1. СИНХРОННАЯ ЗАГРУЗКА ДАННЫХ ЧЕРЕЗ APIMOEX
# ==========================================
import requests
import apimoex
import pandas as pd

def load_moex_data():
    print("Загрузка данных по акциям Сбербанка (SBER) через apimoex...")

    with requests.Session() as session:
        # apimoex.get_board_history собирает все исторические данные за период.
        # ВАЖНО: если строк окажется больше 100, apimoex автоматически сделает
        # постраничные запросы (limit=100) и склеит результат.
        data = apimoex.get_board_history(
            session,
            security='SBER',
            start='2023-01-01',
            end='2026-01-01',
            board='TQBR',
            columns=('TRADEDATE', 'SECID', 'CLOSE', 'VOLUME', 'OPEN', 'HIGH', 'LOW'),
        )
        return pd.DataFrame(data)

# Синхронный вызов — работает и в скрипте, и в Jupyter Notebook
df = load_moex_data()

# Предобработка таблицы данных
df['TRADEDATE'] = pd.to_datetime(df['TRADEDATE'])
df = df.sort_values('TRADEDATE').dropna(subset=['CLOSE']).reset_index(drop=True)

prices = df['CLOSE'].values.reshape(-1, 1)
print(f"Данные успешно загружены! Найдено торговых дней: {len(df)}")


In [ ]:
# ==========================================
# 2. ПРЕДОБРАБОТКА ДАННЫХ И СКОЛЬЗЯЩЕЕ ОКНО
# ==========================================
# scaler = MinMaxScaler(feature_range=(0, 1))
# scaled_prices = scaler.fit_transform(prices)

def create_dataset(dataset, look_back=20):
    X, y = [], []
    for i in range(len(dataset) - look_back):
        X.append(dataset[i:(i + look_back), 0])
        y.append(dataset[i + look_back, 0])
    return np.array(X), np.array(y)

LOOK_BACK = 20
X, y = create_dataset(prices, LOOK_BACK)

# Разделение на train/test (80% / 20%)
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# В PyTorch для Conv1D нужен формат тензора: [batch_size, channels, time_steps]
# У нас 1 признак (цена закрытия), поэтому channels = 1
X_train_t = torch.FloatTensor(X_train).unsqueeze(1) 
y_train_t = torch.FloatTensor(y_train).unsqueeze(1)
X_test_t = torch.FloatTensor(X_test).unsqueeze(1)
y_test_t = torch.FloatTensor(y_test).unsqueeze(1)

# Создание DataLoader для обучения батчами
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=16, shuffle=False)

In [ ]:
# ==========================================
# 3. АРХИТЕКТУРА МОДЕЛИ 1D CNN НА PYTORCH
# ==========================================
class CNN1DTimeSeries(nn.Module):
    def __init__(self, look_back):
        super(CNN1DTimeSeries, self).__init__()
        # Первый слой: входные каналы=1, выходные=64, размер ядра=3
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=64, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool1d(kernel_size=2)
        
        # Второй слой: входные каналы=64, выходные=32, размер ядра=3
        self.conv2 = nn.Conv1d(in_channels=64, out_channels=32, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool1d(kernel_size=2)
        
        # Расчет размера вектора после сверток и пулингов для Flatten
        # look_back (20) -> pool1 (10) -> pool2 (5)
        flatten_features = 32 * (look_back // 4) 
        
        # Полносвязные слои
        self.fc1 = nn.Linear(flatten_features, 50)
        self.relu3 = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.fc2 = nn.Linear(50, 1)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = x.view(x.size(0), -1) # Сплющивание (Flatten)
        x = self.dropout(self.relu3(self.fc1(x)))
        x = self.fc2(x)
        return x

model = CNN1DTimeSeries(LOOK_BACK)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [ ]:
# ==========================================
# 4. ЦИКЛ ОБУЧЕНИЯ (TRAINING LOOP)
# ==========================================
print("\nОбучение модели в PyTorch...")
EPOCHS = 30
model.train()

for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()       # Сброс градиентов
        predictions = model(batch_X) # Прямой проход
        loss = criterion(predictions, batch_y) # Расчет ошибки
        loss.backward()             # Обратный проход (backpropagation)
        optimizer.step()            # Обновление весов
        epoch_loss += loss.item() * batch_X.size(0)
    
    total_epoch_loss = epoch_loss / len(train_loader.dataset)
    if (epoch + 1) % 5 == 0:
        print(f"Эпоха {epoch+1}/{EPOCHS}, Loss (MSE): {total_epoch_loss:.6f}")


In [ ]:
# ==========================================
# 5. ОЦЕНКА И INFERENCE
# ==========================================
model.eval() # Перевод в режим валидации (выключает Dropout)
with torch.no_grad():
    test_predict = model(X_test_t).numpy()

# ==========================================
# 6. ВИЗУАЛИЗАЦИЯ 
# ==========================================
plt.figure(figsize=(12, 6))
plt.plot(df['TRADEDATE'].values[LOOK_BACK:train_size+LOOK_BACK], y_train.reshape(-1, 1), label='Обучающая выборка (Факт)')
plt.plot(df['TRADEDATE'].values[train_size+LOOK_BACK:], y_test, label='Тестовая выборка (Реальные цены)', color='green')
plt.plot(df['TRADEDATE'].values[train_size+LOOK_BACK:], test_predict, label='Предсказание 1D CNN (PyTorch)', color='red', linestyle='--')
plt.title('Прогнозирование цен акций SBER через PyTorch 1D CNN')
plt.xlabel('Дата')
plt.ylabel('Цена акции (руб.)')
plt.legend()
plt.grid(True)
plt.show()


# Задание на самостоятельную работу

Исследование точности и интерпретируемости прогноза цен финансовых инструментов при заданном горизонте.


## 2. Получение и предобработка данных

### 2.1 Источник данных

Используйте библиотеку `apimoex` для получения исторических дневных свечей через ISS API Московской биржи.

**Рекомендуемые инструменты**:
- Акции на ваш выбор, например: `SBER`, `GAZP`, `LKOH`
- Индекс: `IMOEX` (board `SNDX`)

### 2.2 Временной диапазон

Выберите не менее **3 лет** дневных данных (например, с 2022-01-01 по 2025-01-01), чтобы модель могла выучить тренды и сезонные паттерны.

### 2.3 Требования к предобработке

**Обязательное разделение** на обучающую, валидационную и тестовую выборки. Тестовая выборка — последний по времени отрезок. Случайное перемешивание запрещено.


## 3. Реализация моделей

Необходимо сравнить **две модели** из списка ниже.

### Модель A: N-BEATS (интерпретируемая архитектура на базе MLP)

**Ключевое требование**: использовать стеки `Trend` и `Seasonality`, задействовав механизм двойных остаточных соединений для автоматического разложения ряда на тренд и сезонность.

**Рекомендуемые гиперпараметры**:
- `stack_types=['Trend', 'Seasonality']`
- `n_blocks_per_stack=3`
- `thetas_dim` (порядок полинома тренда) — 3
- `backcast_length` (окно ретроспективы) — в 3–5 раз больше горизонта прогноза

**На что обратить внимание**:
- Можно использовать реализацию из `pytorch-forecasting` или `darts`.
- В процессе обучения контролируйте качество восстановления входа по `backcast` — это уникальный самоконтролируемый сигнал N-BEATS.

### Модель B: ConvTimeNet (чисто свёрточная многомасштабная архитектура)

**Ключевое требование**: использовать **деформируемые патчи** и **иерархические чисто свёрточные блоки** для улавливания локальных паттернов и масштабных зависимостей.

**Рекомендуемые гиперпараметры**:
- `context_window` согласован с горизонтом (например, прогноз 5 дней — ретроспектива 20 дней)
- `patch_ks` (размер патча) — 4–8
- `dw_ks` (размеры depthwise-свёрток) — многомасштабный набор, например `(9, 3)`
- `deformable=True` для адаптивного восприятия локальных паттернов

**На что обратить внимание**:
- Можно использовать `ConvTimeNetForecaster` из `sktime`.
- Соблюдайте ограничение `patch_ks ≤ context_window`.



## 4. Оценка и сравнение

### 4.1 Метрики

Рассчитайте на тестовой выборке и сведите в таблицу:

| Метрика | Смысл | Особенности в финансах |
|---------|-------|------------------------|
| **MAE** | Средняя абсолютная ошибка | Устойчива к выбросам, не отражает направление |
| **RMSE** | Корень из средней квадратичной ошибки | Сильнее штрафует крупные промахи |
| **MAPE** | Средняя абсолютная процентная ошибка | Нестабильна при ценах, близких к нулю |
| **WAPE** | Взвешенная абсолютная процентная ошибка | Устойчива к нулям, подходит для объёмов |

**Дополнительно**: рассчитайте **точность направления** (доля верно предсказанных знаков изменения) — ключевая метрика для финансовых приложений.

### 4.2 Визуализация

- График **прогноз vs факт** на тестовой выборке.
- Для N-BEATS — **отдельные графики трендовой и сезонной компонент**, чтобы проверить, что модель выучила осмысленные паттерны.
- График **ошибки во времени** для выявления структурных сдвигов (например, смены режима).




## 5. Анализ и обсуждение


1. **Интерпретируемость**: соответствует ли трендовая компонента N-BEATS реальной долгосрочной динамике инструмента? Улавливает ли сезонная компонента внутринедельные или внутримесячные эффекты? Обладает ли ConvTimeNet аналогичной способностью к разложению?

2. **Точность vs эффективность**: подтверждается ли заявление ConvTimeNet о «линейной сложности и меньшем потреблении ресурсов»? Насколько быстрее он обучается и сколько экономит памяти? Какова цена в точности?

3. **Устойчивость к структурным сдвигам**: выберите тестовый интервал с выраженной сменой режима (например, резкие колебания февраля–марта 2022 года). Какая модель лучше адаптируется к таким изменениям?

4. **Анализ ошибок**: найдите точки с наибольшей ошибкой и объясните возможные причины (гэпы, новостные события, падение ликвидности).
